# 📦 DVC Pipeline Example

Run an `mlprep` pipeline from a DVC stage for reproducible and versioned data processing.

## Scenario
- Input: `raw_data.csv`
- Pipeline: Clean data with cast, fill_null, drop_null, filter, sort
- Output: `outputs/clean_users.parquet` tracked by DVC

In [ ]:
!pip install mlprep-rust dvc -q

In [ ]:
import pandas as pd
import numpy as np
import os

os.makedirs('outputs', exist_ok=True)

data = {
    'id': [1, 2, 3, 4, 5],
    'name': ['Alice', 'Bob', 'Charlie', None, 'Eve'],
    'age': ['25', '17', '30', '22', None],
    'city': ['Tokyo', 'Osaka', None, 'Nagoya', 'Fukuoka'],
    'income': [50000, None, 70000, 60000, 55000]
}
df = pd.DataFrame(data)
df.to_csv('raw_data.csv', index=False)
print('Generated raw_data.csv')
df

In [ ]:
pipeline_yaml = '''name: dvc_mlprep_pipeline
inputs:
  - path: raw_data.csv
    format: csv

steps:
  - type: cast
    columns:
      age: Int64
      income: Float64
  - type: fill_null
    columns: [income]
    strategy: mean
  - type: drop_null
    columns: [city]
  - type: filter
    condition: "age >= 18"
  - type: sort
    by: [age]

outputs:
  - path: outputs/clean_users.parquet
    format: parquet
    compression: zstd
'''
with open('pipeline.yaml', 'w') as f:
    f.write(pipeline_yaml)
print(pipeline_yaml)

In [ ]:
dvc_yaml = '''stages:
  preprocess:
    desc: Run mlprep pipeline and track the processed dataset
    wdir: .
    cmd: mlprep run pipeline.yaml
    deps:
      - pipeline.yaml
      - raw_data.csv
    outs:
      - outputs/clean_users.parquet
'''
with open('dvc.yaml', 'w') as f:
    f.write(dvc_yaml)
print('Created dvc.yaml')
print(dvc_yaml)

## Run mlprep directly (without DVC)

In [ ]:
!mlprep run pipeline.yaml

In [ ]:
if os.path.exists('outputs/clean_users.parquet'):
    result = pd.read_parquet('outputs/clean_users.parquet')
    print(f'Output: {len(result)} rows')
    print(result)
else:
    print('Output not found')

## Run with DVC

In a git-initialized project, run:
```bash
dvc init
dvc repro
dvc status
```

In [ ]:
print('📦 DVC + mlprep Integration')
print('='*40)
print('1. dvc init: Initialize DVC in git repo')
print('2. dvc repro: Run the mlprep pipeline')
print('3. dvc push: Push processed data to remote')
print('\nBenefits: Reproducibility, versioning, caching')